# Quick-Start: Tabular Attribution with Captum

**Goal:** Explain any tabular PyTorch model prediction in under 2 minutes.

**What you get:** Per-feature importance bar chart using three methods (IG, FA, Shapley).

**Prerequisites:**
```bash
pip install captum scikit-learn
```

---

In [ ]:
# ============================================================
# STEP 1: Your Model and Data
# ============================================================
import torch
import torch.nn as nn
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import spearmanr

from captum.attr import IntegratedGradients, FeatureAblation, ShapleyValueSampling

torch.manual_seed(42)

# --- CONFIGURE HERE: swap for your own model and data ---

# Option A: Use UCI Wine Quality (auto-downloaded)
import urllib.request, io
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
import torch.optim as optim

WINE_URL = "https://archive.ics.uci.edu/ml/machine-learning-databases/wine-quality/winequality-red.csv"
try:
    with urllib.request.urlopen(WINE_URL, timeout=15) as resp:
        df = pd.read_csv(io.BytesIO(resp.read()), sep=';')
except Exception:
    print("Using synthetic fallback.")
    n = 1599
    df = pd.DataFrame({
        c: np.random.randn(n)
        for c in ['fixed acidity','volatile acidity','citric acid','residual sugar',
                   'chlorides','free sulfur dioxide','total sulfur dioxide','density',
                   'pH','sulphates','alcohol']
    })
    df['quality'] = np.random.choice(range(3, 9), n)

# --- CONFIGURE HERE: your feature names ---
FEATURE_NAMES = [
    'fixed_acidity', 'volatile_acidity', 'citric_acid',
    'residual_sugar', 'chlorides', 'free_SO2',
    'total_SO2', 'density', 'pH', 'sulphates', 'alcohol'
]

X = df.drop('quality', axis=1).values.astype(np.float32)
y = (df['quality'].values >= 6).astype(np.float32)

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y, test_size=0.2, random_state=42, stratify=y
)

X_train_t = torch.tensor(X_train, dtype=torch.float32)
X_test_t  = torch.tensor(X_test,  dtype=torch.float32)
y_train_t = torch.tensor(y_train, dtype=torch.float32)

# Training mean as baseline
baseline = torch.tensor(X_train.mean(axis=0), dtype=torch.float32).unsqueeze(0)

print(f"Data: {X_train.shape[0]} train, {X_test.shape[0]} test, {len(FEATURE_NAMES)} features")


# --- CONFIGURE HERE: your model architecture ---
class TabularModel(nn.Module):
    def __init__(self, n_features, n_classes=2):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(n_features, 64), nn.ReLU(), nn.Dropout(0.2),
            nn.Linear(64, 32), nn.ReLU(),
            nn.Linear(32, n_classes)
        )
    def forward(self, x):
        return self.net(x)


N_FEATURES = len(FEATURE_NAMES)
N_CLASSES  = 2

model = TabularModel(N_FEATURES, N_CLASSES)
optimizer = optim.Adam(model.parameters(), lr=0.001)
criterion = nn.CrossEntropyLoss()

# Train
model.train()
for _ in range(80):
    loss = criterion(model(X_train_t), y_train_t.long())
    optimizer.zero_grad(); loss.backward(); optimizer.step()

model.eval()
with torch.no_grad():
    acc = (model(X_test_t).argmax(1) == torch.tensor(y_test).long()).float().mean()
print(f"Model ready. Test accuracy: {acc:.1%}")

In [ ]:
# ============================================================
# STEP 2: Pick a Sample to Explain
# ============================================================

with torch.no_grad():
    test_probs = torch.softmax(model(X_test_t), dim=1)

# --- CONFIGURE HERE: pick any sample index, or use the most confident one ---
SAMPLE_IDX = test_probs[:, 1].argmax().item()  # Most confident "good" prediction
TARGET_CLASS = 1  # 1 = "good wine"

sample = X_test_t[SAMPLE_IDX].unsqueeze(0)
pred_prob = test_probs[SAMPLE_IDX, TARGET_CLASS].item()
pred_label = 'Good wine' if TARGET_CLASS == 1 else 'Poor wine'

print(f"Explaining sample #{SAMPLE_IDX}: {pred_label} (P={pred_prob:.1%})")

# Show original feature values
sample_orig = scaler.inverse_transform(sample.numpy()).squeeze()
print("Feature values:")
for name, val in zip(FEATURE_NAMES, sample_orig):
    print(f"  {name:20s}: {val:.3f}")

In [ ]:
# ============================================================
# STEP 3: Run Three Attribution Methods
# ============================================================

# Integrated Gradients
ig = IntegratedGradients(model)
ig_attr, ig_delta = ig.attribute(
    sample.clone().requires_grad_(True),
    baselines=baseline, target=TARGET_CLASS,
    n_steps=100, return_convergence_delta=True
)
ig_vals = ig_attr.squeeze().detach().numpy()
print(f"IG  convergence delta: {ig_delta.item():.5f}")

# Feature Ablation
fa = FeatureAblation(model)
fa_attr = fa.attribute(sample, baselines=baseline, target=TARGET_CLASS)
fa_vals = fa_attr.squeeze().detach().numpy()
print("FA: done")

# Shapley Value Sampling
svs = ShapleyValueSampling(model)
svs_attr = svs.attribute(sample, baselines=baseline, target=TARGET_CLASS, n_samples=200)
svs_vals = svs_attr.squeeze().detach().numpy()
print("Shapley (n_samples=200): done")

# Completeness checks
with torch.no_grad():
    f_x   = model(sample)[0, TARGET_CLASS].item()
    f_bl  = model(baseline)[0, TARGET_CLASS].item()
total_diff = f_x - f_bl

print(f"\nCompleteness verification (sum should ≈ {total_diff:.5f}):")
print(f"  IG  sum: {ig_vals.sum():.5f}  (delta={ig_delta.item():.5f})")
print(f"  FA  sum: {fa_vals.sum():.5f}")
print(f"  SVS sum: {svs_vals.sum():.5f}")

In [ ]:
# ============================================================
# STEP 4: Visualize
# ============================================================

def normalize(arr):
    m = np.abs(arr).max() + 1e-8
    return arr / m

ig_n  = normalize(ig_vals)
fa_n  = normalize(fa_vals)
svs_n = normalize(svs_vals)

# Sort by Shapley importance
sort_idx = np.argsort(np.abs(svs_n))[::-1]
pos = np.arange(len(FEATURE_NAMES))
w = 0.27

fig, axes = plt.subplots(1, 2, figsize=(18, 6))
fig.suptitle(
    f'Tabular Attribution — {pred_label} (P={pred_prob:.1%})\n'
    f'Sample #{SAMPLE_IDX} | Baseline: training mean',
    fontsize=12
)

# Method comparison (sorted by Shapley)
axes[0].bar(pos - w, ig_n[sort_idx],  w, label='IG',     color='#3498db', alpha=0.85)
axes[0].bar(pos,     fa_n[sort_idx],  w, label='FA',     color='#e74c3c', alpha=0.85)
axes[0].bar(pos + w, svs_n[sort_idx], w, label='Shapley', color='#9b59b6', alpha=0.85)
axes[0].set_xticks(pos)
axes[0].set_xticklabels(
    [FEATURE_NAMES[i][:12] for i in sort_idx],
    rotation=45, ha='right'
)
axes[0].axhline(0, color='black', lw=0.8)
axes[0].set_ylabel('Normalized Attribution')
axes[0].set_title('IG vs Feature Ablation vs Shapley')
axes[0].legend()

# Feature values vs attribution
colors_svs = ['#2ecc71' if v >= 0 else '#e74c3c' for v in svs_n[sort_idx]]
ax2 = axes[1]
bars = ax2.barh(
    [FEATURE_NAMES[i] for i in sort_idx],
    svs_n[sort_idx],
    color=colors_svs[::-1], alpha=0.85  # Reverse for bottom-to-top display
)
ax2.axvline(0, color='black', lw=0.8)
ax2.set_xlabel('Normalized Shapley Attribution')
ax2.set_title('Shapley Attribution (sorted)\nGreen=supports, Red=opposes prediction')

# Annotate with actual feature values
for bar, feat_idx in zip(bars, sort_idx[::-1]):
    ax2.text(
        bar.get_width() + 0.01, bar.get_y() + bar.get_height() / 2,
        f'{sample_orig[feat_idx]:.2f}',
        va='center', fontsize=7, color='gray'
    )

plt.tight_layout()
plt.savefig('tabular_attribution.png', dpi=150, bbox_inches='tight')
plt.show()
print("Saved: tabular_attribution.png")

In [ ]:
# ============================================================
# STEP 5: Method Agreement Summary
# ============================================================

rho_ig_fa,  _ = spearmanr(ig_n, fa_n)
rho_ig_svs, _ = spearmanr(ig_n, svs_n)
rho_fa_svs, _ = spearmanr(fa_n, svs_n)

print("Attribution Summary")
print("=" * 45)
print(f"Sample #{SAMPLE_IDX}: {pred_label} (P={pred_prob:.1%})")
print(f"IG convergence delta: {ig_delta.item():.5f}",
      "OK" if abs(ig_delta.item()) < 0.05 else "WARNING")
print()
print(f"Top features by Shapley (absolute):")
for rank, i in enumerate(sort_idx[:5]):
    direction = 'supports' if svs_vals[i] >= 0 else 'opposes'
    print(f"  #{rank+1} {FEATURE_NAMES[i]:20s}: {svs_vals[i]:+.5f} ({direction})")
print()
print("Method agreement (Spearman ρ):")
print(f"  IG vs FA:      ρ = {rho_ig_fa:.3f}")
print(f"  IG vs Shapley: ρ = {rho_ig_svs:.3f}")
print(f"  FA vs Shapley: ρ = {rho_fa_svs:.3f}")
mean_rho = np.mean([rho_ig_fa, rho_ig_svs, rho_fa_svs])
print()
if mean_rho > 0.7:
    print(f"High agreement (mean ρ={mean_rho:.2f}): attributions are robust.")
elif mean_rho > 0.4:
    print(f"Moderate agreement (mean ρ={mean_rho:.2f}): generally consistent.")
else:
    print(f"Low agreement (mean ρ={mean_rho:.2f}): possible feature interactions.")
    print("Use Shapley values — they capture interactions IG/FA may miss.")

print("\nQuick-start complete. Copy this notebook into your project.")